[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part3_agents/ch09_langgraph.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 9장 에이전트 워크플로: LangGraph
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제3부 에이전트 — 스스로 일하는 LLM
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
!pip install -q google-genai langgraph langchain-google-genai

### API 키와 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.0-flash"   # 모델 갱신 시 이 한 줄만 변경

## 예제 9-1. 상태 그래프 에이전트

In [ ]:
def search_node(state):                 # 노드: 검색
    state["found"] = retrieve(state["question"], k=1)
    state["tries"] += 1
    return state

def answer_node(state):                 # 노드: 답 생성
    prompt = f"근거: {state['found'][0]}\n질문: {state['question']}\n근거로만 답하라."
    state["answer"] = client.models.generate_content(model=GEMINI_FLASH, contents=prompt).text
    return state

def run(question, max_tries=3):
    state = {"question": question, "found": None, "tries": 0, "answer": None}
    node = "search"
    while node != "end":
        if node == "search":
            state = search_node(state)
            # 조건부 선: 찾았으면 답으로, 아니면 한계까지 다시 검색
            node = "answer" if state["found"] else ("search" if state["tries"] < max_tries else "answer")
        elif node == "answer":
            state = answer_node(state)
            node = "end"
    return state["answer"]

print(run("환불 며칠 안에 돼?"))

## 예제 9-2. 컨텍스트 요약으로 관리

In [ ]:
def trim_context(history, keep=2):
    if len(history) <= keep:
        return history
    summary = f"(앞 {len(history) - keep}단계 요약) " + " / ".join(history[:-keep])
    return [summary] + history[-keep:]

steps = ["검색 결과", "계산 결과", "초안 작성", "검토", "수정"]
for line in trim_context(steps):
    print(line)

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 9장 노트북